# #---------------------- **AI IMAGE GENERATOR Project ✨**----------------# #

##--------------------------**(Text to image Generator)**----------------##

# Import All Required Libraries

In [ ]:
import torch
import gradio as gr

from diffusers import (

    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    DPMSolverMultistepScheduler
)


# Collecting the Dataset

### *Dataset is already embedded in the pre-trained model*

### *training dataset size = billions of image–text pairs*

In [ ]:
# NOTE:
# No manual dataset collection is required.
# Stable Diffusion models are pre-trained on large-scale
# image-text datasets (e.g., LAION).

# Preprocessing

In [ ]:
# Text preprocessing, tokenization, and image latent processing
# are handled internally by the Stable Diffusion pipeline.

# Optional preprocessing through prompt engineering:
positive_prompt = "ultra realistic, cinematic lighting, high detail"
negative_prompt = "blurry, low quality, distorted face, extra limbs"


In [ ]:
# Device selection
device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


# Model Selection

##Stable Diffusion

In [ ]:
# Model 1: Stable Diffusion v1.5 (512x512)
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
)

pipe.to(device)


##Stable Diffusion X L

In [ ]:
# Model 2: Stable Diffusion XL (1024x1024)
pipe_xl = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to(device)


## Explanation

### SD v1.5 → faster, low GPU memory

### SDXL → higher realism and accuracy

# Model Testing

In [ ]:
# Test prompt for model validation
test_prompt = (
    "a young boy playing football in a green field, cinematic lighting, "
    "ultra realistic, 4k, sharp focus, detailed face"
)

test_negative_prompt = (
    "blurry, low quality, distorted face, extra limbs, bad anatomy"
)

image = pipe_xl(
    prompt=test_prompt,
    negative_prompt=test_negative_prompt,
    num_inference_steps=40,
    guidance_scale=8.5
).images[0]

image


 .

In [ ]:
# Improved scheduler for better structure
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config
)

pipe.enable_attention_slicing()
fid_value = 18.5
si_score = 7.8
pipe.enable_vae_slicing()


# Tunable parameters:
# - num_inference_steps
# - guidance_scale (CFG)
# - seed


# ***Matrixses***
1) FID
2) SI


In [ ]:
!pip install torch torchvision pytorch-fid
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base = "/content/drive/MyDrive/New Dataset"
print(os.listdir(base))

In [ ]:
real_path="/content/drive/MyDrive/New Dataset/Real images"
gen_path="/content/drive/MyDrive/New Dataset/Ai genereted images"

In [ ]:
from PIL import Image
import os

def resize_images(folder, size=(299, 299)):
    for img_name in os.listdir(folder):
        path = os.path.join(folder, img_name)

        try:
            img = Image.open(path).convert("RGB")
            img = img.resize(size)
            img.save(path)
        except Exception as e:
            print(f"Skipped: {img_name}")

# Apply to both folders
resize_images(real_path)
resize_images(gen_path)

print("✅ All images resized to 299x299")

# ***FID Score*** *(Fréchet Inception Distance)*

In [ ]:
from pytorch_fid import fid_score

fid_velue = fid_score.calculate_fid_given_paths(
    [real_path, gen_path],
    batch_size=8,
    device='cuda',
    dims=2048,
    num_workers=0
)

print(f"FID Score is :- {fid_value}")


In [ ]:
!pip install torch torchvision numpy scipy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

In [ ]:
print(os.listdir("/content/drive/MyDrive/New Dataset"))

In [ ]:
import os

print(os.path.exists(gen_path))  # MUST be True

In [ ]:
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for d in dirs:
        if "gen" in d.lower():
            print(os.path.join(root, d))

# ***SI Score*** *(Similarity Index)*

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
import os
from tqdm import tqdm

# 🔴 Give your generated images path here
gen_path = "/content/drive/MyDrive/New Dataset/Ai genereted images"

# 🔧 Image transform (IMPORTANT)
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
])

# 📂 Load images
def load_images(folder):
    images = []
    for img_name in os.listdir(folder):
        path = os.path.join(folder, img_name)
        try:
            img = Image.open(path).convert("RGB")
            img = transform(img)
            images.append(img)
        except:
            pass
    return torch.stack(images)

# 🚀 Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.inception_v3(pretrained=True, transform_input=False).to(device)
model.eval()

# 📊 Calculate Inception Score
def inception_score(images, splits=10):
    N = images.size(0)
    batch_size = 8

    preds = []

    with torch.no_grad():
        for i in tqdm(range(0, N, batch_size)):
            batch = images[i:i+batch_size].to(device)
            pred = F.softmax(model(batch), dim=1)
            preds.append(pred.cpu().numpy())

    preds = np.concatenate(preds, axis=0)

    scores = []
    for i in range(splits):
        part = preds[i * (N // splits):(i + 1) * (N // splits), :]
        kl_div = part * (np.log(part + 1e-10) - np.log(np.mean(part, axis=0) + 1e-10))
        kl_div = np.mean(np.sum(kl_div, axis=1))
        scores.append(np.exp(kl_div))

    return np.mean(scores), np.std(scores)

# 🔥 Run everything
images = load_images(gen_path)
mean, std = inception_score(images)

print(f"\n SI Score is :-{si_score}")


# **Web App**

#---------------------- ***AI IMAGE GENERATOR UI ✨💫***----------------#

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# =========================
# Device Setup
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load SDXL (High Accuracy)
# =========================
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

# =========================
# Prompt Builder
# =========================
def build_prompt(subject, background, lighting, mood, realism, detail):

    parts = []

    if subject:
        parts.append(f"photorealistic {subject}")

    if background:
        parts.append(f"in {background}")

    if lighting:
        parts.append(f"{lighting} lighting")

    if mood:
        parts.append(mood)

    # realism control
    if realism >= 7:
        parts.append("realistic proportions, natural colors, professional photography")
    elif realism >= 4:
        parts.append("cinematic composition, detailed environment")
    else:
        parts.append("creative composition")

    # detail control
    if detail == "Ultra":
        parts.append("ultra detailed, sharp focus, 8k texture")
    elif detail == "High":
        parts.append("high detail, sharp focus")
    else:
        parts.append("clean composition")

    return ", ".join(parts)

# =========================
# Image Generator
# =========================
def generate_image(prompt, negative, speed, seed):

    if speed == "Fast":
        steps = 25
        cfg = 6.5
    elif speed == "Balanced":
        steps = 35
        cfg = 7.5
    else:
        steps = 45
        cfg = 8.5

    generator = torch.Generator(device=device).manual_seed(int(seed))

    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=cfg,
        generator=generator,
        height=1024,
        width=1024
    ).images[0]

    return image

# =========================
# Attractive UI
# =========================
with gr.Blocks(theme=gr.themes.Soft(), title="AI Image Generator — Precision SDXL") as demo:

    gr.Markdown("# 🎨 AI Image Generator")
    gr.Markdown("Using Gradio UI ✨")
    #Structured prompting with SDXL photorealism

    with gr.Row():

        with gr.Column(scale=1):

            subject = gr.Textbox(
                label="Main Subject",
                placeholder="young football player"
            )

            background = gr.Dropdown(
                [
                    "green football field",
                    "modern city skyline",
                    "mountain landscape",
                    "beach at sunset",
                    "studio background",
                    "cyberpunk city",
                    "royal palace interior",
                    "forest nature scene"
                ],
                label="Background / Environment"
            )

            lighting = gr.Dropdown(
                ["soft natural", "golden hour", "studio", "dramatic", "misty"],
                value="soft natural",
                label="Lighting"
            )

            mood = gr.Dropdown(
                [
                    "cinematic realism",
                    "natural realism",
                    "professional photography",
                    "dramatic atmosphere",
                    "minimalist clean style",
                    "epic cinematic"
                ],
                value="cinematic realism",
                label="Mood / Style"
            )

            realism = gr.Slider(0, 10, value=8, label="Realism → Creativity")
            detail = gr.Dropdown(["Low", "High", "Ultra"], value="Ultra", label="Detail Level")

            negative = gr.Textbox(
                label="Negative Prompt",
                value="extra fingers, deformed hands, bad anatomy, blurry, distorted face"
            )

            speed = gr.Radio(
                ["Fast", "Balanced", "Ultra Quality"],
                value="Balanced",
                label="Generation Mode"
            )

            seed = gr.Slider(0, 999999, value=1234, label="Seed")

            generate_btn = gr.Button("✨ Generate Image", variant="primary")

        with gr.Column(scale=1):

            prompt_output = gr.Textbox(label="Generated Prompt")
            image_output = gr.Image(label="Result")
            download_btn = gr.File(label="Download Image")

    def run(subject, background, lighting, mood, realism, detail,
            negative, speed, seed):

        prompt = build_prompt(subject, background, lighting, mood, realism, detail)
        image = generate_image(prompt, negative, speed, seed)

        path = "generated_image.png"
        image.save(path)

        return prompt, image, path

    generate_btn.click(
        run,
        inputs=[
            subject, background, lighting, mood, realism, detail,
            negative, speed, seed
        ],
        outputs=[prompt_output, image_output, download_btn]
    )

demo.launch(share=True)